# TexTable: report tables

<style>
blockquote:has(.notebook-admonition-title) {
  --notebook-admonition-color: var(--color-admonition-title--note, #087fc7);
  --notebook-admonition-title-background:
    var(--color-admonition-title-background--note, rgba(8, 127, 199, 0.18));
  background: var(--color-admonition-background, transparent);
  border: 0;
  border-left: 0.2rem solid var(--notebook-admonition-color);
  border-radius: 0.2rem;
  box-shadow: 0 0.2rem 0.5rem rgba(0, 0, 0, 0.05), 0 0 0.0625rem rgba(0, 0, 0, 0.1);
  font-size: var(--admonition-font-size, 0.8125rem);
  margin: 1rem auto;
  overflow: hidden;
  padding: 0 0.5rem 0.5rem;
}
blockquote p:has(> .notebook-admonition-title) {
  background: var(--notebook-admonition-title-background);
  font-size: var(--admonition-title-font-size, 0.8125rem);
  font-weight: 500;
  line-height: 1.3;
  margin: 0 -0.5rem 0.5rem;
  padding: 0.4rem 0.5rem 0.4rem 2rem;
  position: relative;
}
blockquote p:has(> .notebook-admonition-title)::before {
  color: var(--notebook-admonition-color);
  content: "✎";
  left: 0.65rem;
  position: absolute;
}
.notebook-admonition-title {
  font-weight: inherit;
}
table:not(.dataframe) {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
  border-collapse: collapse;
}
table:not(.dataframe) th,
table:not(.dataframe) td {
  border: 1px solid var(--docs-hairline, rgba(128, 128, 128, 0.35));
}
</style>

`TexTable` builds a conventional LaTeX `tabular` from a DataFrame. The TeX source is
generated in Python, so inspecting it needs no compiler; previewing and exporting PDF
or PNG use a local LaTeX installation. The rendered images on this page are committed
documentation assets produced from exactly the code shown, so ordinary documentation
builds never invoke TeX.

**Why two table classes?** `TexTable` and `TikzTable` share this exact builder API;
they differ only in the TeX dialect they emit. `TexTable` produces a plain `tabular`
with minimal dependencies and one compile pass, so its output pastes into any report or
journal template, and it is the right default. The
[TikzTable guides](tikztable.ipynb) cover the `nicematrix` dialect, which costs
two compile passes but adds per-cell borders and free-form drawing. Because the APIs
match, switching dialects later is a one-word change.

In [ ]:
import numpy as np
import pandas as pd

districts = pd.DataFrame(
    {
        "District": [f"CD {i}" for i in range(1, 9)],
        "BVAP share": [0.12, 0.18, 0.22, 0.31, 0.38, 0.44, 0.52, 0.58],
        "Dem share": [0.35, 0.41, 0.44, 0.47, 0.50, 0.55, 0.61, 0.66],
        "Polsby-Popper": [0.18, 0.22, 0.27, 0.31, 0.33, 0.35, 0.41, 0.44],
        "Pop. deviation": [0.004, 0.002, np.nan, 0.006, 0.001, 0.008, 0.003, 0.005],
    }
)
districts

## The default table

Defaults are meant to be readable without setup: bold headers, a double rule below the
header row, centered values.

In [ ]:
from gerrytools.latex import TexTable

table = TexTable(districts)
table.set_decimal_count(3)
print(table)

![Default district table][textable-default]

[textable-default]: ../../_static/images/latex/textable-default.png

## Copy it straight into a document

What `print(table)` shows *is* the complete standalone document: `str()` and the
notebook `repr` both return `table.document.to_tex()`, so pasting the printed output
into an empty `.tex` file compiles as-is. The preamble stays minimal, with packages
added dynamically as table options and formatters require them, so the source carries
only what it uses.

`print_table()` prints the table environment alone, for pasting into an existing
report, and `preview()`, `document.save_pdf()`, and `document.save_png()` compile the
document for you.

In [ ]:
table.print_table()

## Index columns and rules

`include_index()` adds the DataFrame index as a leading column. Vertical rules attach
to column boundaries and horizontal rules to row boundaries, each with a count.

In [ ]:
table = TexTable(districts)
table.set_decimal_count(2)
table.include_index(name="Row", alignment="c")
table.add_vrule_right_of(0)
table.add_hrule_above([4])
print(table)

![Index column with added rules][textable-rules]

[textable-rules]: ../../_static/images/latex/textable-rules.png

## Header groups

`set_header_groups()` spans a labeled group row over related columns, and
`set_column_headers_text_format()` controls the header text style.

In [ ]:
table = TexTable(districts)
table.set_decimal_count(2)
table.set_header_groups(
    {
        "Identity": ["District"],
        "Demographics and votes": ["BVAP share", "Dem share"],
        "Diagnostics": ["Polsby-Popper", "Pop. deviation"],
    }
)
table.set_column_headers_text_format(bold=False, italic=True)
print(table)

![Grouped headers with italic column names][textable-groups]

[textable-groups]: ../../_static/images/latex/textable-groups.png

## Row highlights

`highlight_rows()` accepts GerryTools color names (with xcolor-style mixes), hex
strings, and RGB tuples.

In [ ]:
table = TexTable(districts)
table.set_decimal_count(2)
table.highlight_rows([0], color="cherryblossompink")
table.highlight_rows([3], color="#c7e5f4")
table.highlight_rows([6], color="amber!40!white")
print(table)

![Three highlighted rows in different colors][textable-highlights]

[textable-highlights]: ../../_static/images/latex/textable-highlights.png

## Formatters

A formatter receives a cell value and returns display text, or wraps the existing
display text. Formatters attach to numbers, strings, columns, rows, or the index, and
`set_nan_string()` controls missing values.

In [ ]:
from gerrytools.latex.formatters import (
    compose_formatters,
    diverging_gradient_formatter,
    highlight_between,
    highlight_ge,
    round_decimals,
    wrap_with_tex_command,
)


def shout(value):
    return value.upper() if isinstance(value, str) else value


table = TexTable(districts)
table.set_nan_string("---")
table.set_string_formatter(shout)
table.set_column_formatter("BVAP share", round_decimals(2))
table.set_column_formatter("Pop. deviation", round_decimals(3))
print(table)

![Uppercased names with per-column rounding][textable-formatters]

[textable-formatters]: ../../_static/images/latex/textable-formatters.png

## Conditional highlights

The `highlight_gt` / `highlight_ge` / `highlight_lt` / `highlight_le` /
`highlight_between` family colors cells by their original numeric value.
`compose_formatters()` runs right to left, so put rounding after the condition: the
condition sees the raw value and the reader sees the rounded one.

In [ ]:
table = TexTable(districts)
table.set_decimal_count(2)
table.set_column_formatter(
    "BVAP share",
    compose_formatters(
        highlight_ge(0.50, color="applegreen!40!white"),
        round_decimals(2),
    ),
)
table.set_column_formatter(
    "Dem share",
    compose_formatters(
        highlight_between(0.45, 0.55, color="amber!35!white", include_lower=False),
        round_decimals(2),
    ),
)
print(table)

![Threshold and interval highlights][textable-conditional]

[textable-conditional]: ../../_static/images/latex/textable-conditional.png

## Gradient cells

`diverging_gradient_formatter()` colors a column through a three-point scale; fix the
bounds to the substantive scale so colors stay comparable across tables. For a whole-
table heatmap, `wrap_with_tex_command()` plus a gradient command from
`gerrytools.latex.commands` keeps the color math in the preamble.

In [ ]:
table = TexTable(districts)
table.set_decimal_count(2)
table.set_column_formatter(
    "Dem share",
    compose_formatters(
        diverging_gradient_formatter(
            lo=0.35,
            mid=0.50,
            hi=0.65,
            color_lo="alizarin",
            color_mid="white",
            color_hi="denim",
            command_name=None,
        ),
        round_decimals(2),
    ),
)
print(table)

![Diverging gradient centered at 50%][textable-diverging]

[textable-diverging]: ../../_static/images/latex/textable-diverging.png

In [ ]:
from gerrytools.latex.commands import tex_twocolor_gradient_command

table = TexTable(districts)
table.set_nan_string("---")
table.set_number_formatter(compose_formatters(wrap_with_tex_command("heatmap"), round_decimals(2)))
table.document.add_command(tex_twocolor_gradient_command("heatmap"))
print(table)

![Whole-table two-color heatmap][textable-heatmap]

[textable-heatmap]: ../../_static/images/latex/textable-heatmap.png

## Customize the document

Each table owns a `TexDocument`. Add packages, package options, commands, and colors
there so the generated preamble stays the single source of truth.

In [ ]:
from gerrytools.latex import TexDocument

document = TexDocument()
document.add_packages("booktabs")
document.add_package_with_options("geometry", ["margin=1in"])
document.add_command(r"\newcommand{\PlanName}{Example Plan}")
document.add_color("reportblue", "denim")
print(document.preamble)

## Related

- [TikzTable basics](tikztable.ipynb) and
  [TikzTable styling](tikztable_styling.ipynb)
- [Paintball plots in LaTeX](paintball.ipynb) and
  [seats-votes plots in LaTeX](seats_votes.ipynb)
- [LaTeX API](../../api/latex.rst)